# DDPM Task 2 — Analysis

Reads precomputed results already synced to `gdrive:lab1-analysis` by `run_analysis.py` and `analyze_deep_dive.py` (both run on a rented GPU pod, not here) plus a couple of lightweight items still read directly from `gdrive:lab1-ckpts` (training-time trajectory PNGs and `train_state.pt` loss histories — both small, no GPU needed). No sampling or FID computation happens in this notebook; it's plotting and table-building only.

1. Table — final FID per config (the graded "best FID" number), all at a common `STEP_CAP` step for comparability
2. Figure — scheduler trajectory comparison (linear/quad/cosine, noise predictor)
3. Figures — predictor sample comparison (noise/x0/mean, linear scheduler): random samples, same-seed samples, and per-timestep reconstruction error
4. Figure + Table — FID vs. training step, all configs, plus convergence-speed table
5. Figure — sample quality progression over training (linear+noise)
6. Figure — training loss curves overlaid, all configs
7. Figure — worst samples per config (least realistic, by nearest-neighbor distance to real Inception features)

**Sample-count note:** the FID curve's non-`STEP_CAP` points use 100 generated samples each (enough to show the trend cheaply across ~40 checkpoints); `STEP_CAP` (100,000 steps, the common reference point across all 5 configs) uses 500 samples (the project's own FID convention, matching the README default) — that's the number in Table 1 and the number that should be screenshotted for submission. The two aren't directly comparable point-for-point; Figure 3 marks the `STEP_CAP` point distinctly for this reason. All 5 configs' training was extended after the original analysis; `STEP_CAP` is used everywhere as the fair comparison point since not every config trained equally far past it (see cell 3's comment for details).

Run cells top to bottom. Requires `lab1-ckpts` and `lab1-analysis` to already be in your Google Drive.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, glob, re
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image

DRIVE_CKPT_ROOT = "/content/drive/MyDrive/lab1-ckpts"      # training-time trajectory PNGs, train_state.pt
ANALYSIS_ROOT = "/content/drive/MyDrive/lab1-analysis"     # run_analysis.py + analyze_deep_dive.py output

# All 5 configs were extended after the original analysis (quad/cosine/x0/mean's real final is
# now step=100000; linear_noise trained further, to 150000). step=100000 is used everywhere
# below as the common reference point across all 5 configs -- comparing each config's own true
# final would conflate "better parameterization/schedule" with "trained for more steps" for
# linear_noise specifically. Its own step=100000 point is a genuine 500-sample number from its
# original (pre-extension) run, not a cheap curve point, so this isn't a downgrade for it either.
STEP_CAP = 100000

# label -> (beta scheduler mode, predictor parameterization)
CONFIGS = {
    "linear_noise": {"mode": "linear", "predictor": "noise"},
    "quad_noise":   {"mode": "quad",   "predictor": "noise"},
    "cosine_noise": {"mode": "cosine", "predictor": "noise"},
    "linear_x0":    {"mode": "linear", "predictor": "x0"},
    "linear_mean":  {"mode": "linear", "predictor": "mean"},
}
PREDICTOR_LABELS = ["linear_noise", "linear_x0", "linear_mean"]

def ckpt_dir(label):
    cfg = CONFIGS[label]
    return f"{DRIVE_CKPT_ROOT}/{label}/predictor_{cfg['predictor']}/beta_{cfg['mode']}"

def latest_traj(label):
    d = ckpt_dir(label)
    files = glob.glob(f"{d}/step=*-traj.png")
    steps = [int(re.search(r"step=(\d+)-traj", f).group(1)) for f in files]
    best = max(steps)
    return f"{d}/step={best}-traj.png", best

def load_png_grid(paths, ax_row):
    """Displays a sorted list of image paths across a row of axes."""
    for ax, path in zip(ax_row, paths):
        ax.imshow(Image.open(path))
        ax.axis("off")
    for ax in ax_row[len(paths):]:
        ax.axis("off")

with open(f"{ANALYSIS_ROOT}/fid_curve_results.json") as f:
    fid_curves_raw = json.load(f)
# {label: {step (int): fid}}, sorted ascending by step
fid_curves = {label: dict(sorted(((int(s), fid) for s, fid in steps.items())))
              for label, steps in fid_curves_raw.items()}

print("Configs:", list(CONFIGS))
print("FID results loaded for:", {l: len(c) for l, c in fid_curves.items()})
print("FID @ step_cap =", STEP_CAP, ":", {l: round(c.get(STEP_CAP, float('nan')), 4) for l, c in fid_curves.items()})

## 1. Table — final FID per config

The graded "best FID" number, all at `STEP_CAP` (100,000 steps) for comparability. quad_noise/cosine_noise/linear_x0/linear_mean's real final checkpoint is exactly this step; linear_noise trained further (to 150,000), but its own step=100,000 point is also a genuine 500-sample number from its original run, not a cheap curve point -- comparing each config's own true final instead would conflate "better parameterization/schedule" with "trained for more steps" specifically for linear_noise. Also doubles as the scheduler/predictor comparison table.

In [ ]:
rows = []
for label, cfg in CONFIGS.items():
    curve = fid_curves[label]
    fid = curve[STEP_CAP]
    true_final_step = max(curve)
    rows.append({"label": label, "scheduler": cfg["mode"], "predictor": cfg["predictor"],
                 "step": STEP_CAP, "FID": fid,
                 "trained_further_to": true_final_step if true_final_step != STEP_CAP else None})
    note = f" (trained further to {true_final_step}, not used here)" if true_final_step != STEP_CAP else ""
    print(f"{label}: step={STEP_CAP} FID={fid:.4f}{note}")

df_final = pd.DataFrame(rows)
df_final

## Noise schedule diagnostics — ᾱ_t and log-SNR(t)

Before comparing trajectories or FID, it helps to see what the three schedules actually *do*: how much clean signal remains at each timestep. No training or checkpoints needed — this is pure closed-form math from the schedule definitions themselves (`T=1000, β₁=1e-4, β_T=0.02`, matching what every config was trained with). Nichol & Dhariwal introduced the cosine schedule specifically because linear destroys signal too quickly in the early-to-mid range for small images; whether that advantage shows up in *this* setup is exactly what the FID comparison below tests.

In [ ]:
T, BETA_1, BETA_T = 1000, 1e-4, 0.02

def alpha_bar_linear():
    betas = np.linspace(BETA_1, BETA_T, T)
    return np.cumprod(1 - betas)

def alpha_bar_quad():
    betas = np.linspace(BETA_1 ** 0.5, BETA_T ** 0.5, T) ** 2
    return np.cumprod(1 - betas)

def alpha_bar_cosine(s=0.008):
    t = np.arange(T + 1, dtype=np.float64)
    f_t = np.cos(((t / T + s) / (1 + s)) * (np.pi / 2)) ** 2
    f_bar = f_t / f_t[0]
    return f_bar[1:]  # ᾱ_t for t=1..T

schedules = {"linear": alpha_bar_linear(), "quad": alpha_bar_quad(), "cosine": alpha_bar_cosine()}
ts = np.arange(1, T + 1)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for name, ab in schedules.items():
    axs[0].plot(ts, ab, label=name)
    log_snr = np.log(ab / (1 - ab))
    axs[1].plot(ts, log_snr, label=name)
axs[0].set_xlabel("t"); axs[0].set_ylabel(r"$\bar\alpha_t$"); axs[0].set_title("Remaining signal fraction")
axs[0].legend(); axs[0].grid(alpha=0.3)
axs[1].set_xlabel("t"); axs[1].set_ylabel("log SNR(t)"); axs[1].set_title("Log signal-to-noise ratio")
axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig0_snr_schedules.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Figure — scheduler trajectory comparison

`linear` vs `quad` vs `cosine` (all noise predictor). Reuses the trajectory strips already rendered during training — note these are from the *last periodic checkpoint before* the final step (e.g. step 98000 not 100000), since trajectory images are only written inside the periodic log_interval block, which never fires exactly at the final step.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 12))
for ax, label in zip(axs, ["linear_noise", "quad_noise", "cosine_noise"]):
    path, step = latest_traj(label)
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title(f"{CONFIGS[label]['mode']} scheduler (trajectory at step {step})", fontsize=14)
plt.tight_layout()
plt.savefig("fig1_scheduler_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Figure — predictor sample comparison

`noise` vs `x0` vs `mean` (all linear scheduler), 8 samples each — precomputed by `run_analysis.py` from each config's final checkpoint and synced to `gdrive:lab1-analysis/predictor_samples/{predictor}/`.

In [ ]:
predictor_labels = ["linear_noise", "linear_x0", "linear_mean"]
fig, axs = plt.subplots(3, 8, figsize=(24, 9.5))

for row, label in enumerate(predictor_labels):
    predictor = CONFIGS[label]["predictor"]
    paths = sorted(glob.glob(f"{ANALYSIS_ROOT}/predictor_samples/{predictor}/*.png"),
                    key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
    load_png_grid(paths, axs[row])
    axs[row, 0].text(-0.3, 0.5, predictor, fontsize=16, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("Predictor comparison: 8 samples each (linear scheduler)", fontsize=16)
plt.tight_layout()
plt.savefig("fig2_predictor_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3b. Figure — same-seed predictor comparison

The grid above uses 8 independent random samples per predictor -- fine for "representative samples," not for "which predictor denoises *this* noise better." This one starts all three predictors from an identical noise sequence (same `torch.manual_seed()` immediately before each model's `.sample()` call -- `model.py`'s `x_T` draw and every reverse step's noise draw are the same shape/count regardless of predictor, so this reproduces the exact same randomness across all three). Differences in the output are now attributable to the parameterization alone. Precomputed by `analyze_deep_dive.py` at `STEP_CAP`, synced to `gdrive:lab1-analysis/same_seed_samples/{predictor}/`.

In [ ]:
fig, axs = plt.subplots(3, 8, figsize=(24, 9.5))
for row, predictor in enumerate(["noise", "x0", "mean"]):
    paths = sorted(glob.glob(f"{ANALYSIS_ROOT}/same_seed_samples/{predictor}/*.png"),
                    key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
    load_png_grid(paths, axs[row])
    axs[row, 0].text(-0.3, 0.5, predictor, fontsize=16, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle(f"Same-seed predictor comparison @ step={STEP_CAP} (identical noise sequence per column)", fontsize=15)
plt.tight_layout()
plt.savefig("fig2b_same_seed_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3c. Figure — per-timestep reconstruction error

FID is a distributional statistic -- it can't say *where* a predictor fails. This asks a sharper question: at a fixed noise level `t`, how well does each predictor recover the real clean image from a genuinely noisy (not generated) input? All three predictors' native output is converted to a common `x_hat_0` using the same closed-form relations `scheduler.py`'s `step_predict_*` use (the mean predictor's conversion properly inverts the posterior-mean formula, not just a stand-in), so the comparison is apples-to-apples regardless of what each network was trained to output directly. Computed by `analyze_deep_dive.py` at `STEP_CAP` against 6 real validation images, at `t/T` ∈ {0.1, 0.3, 0.5, 0.7, 0.9}.

In [ ]:
with open(f"{ANALYSIS_ROOT}/reconstruction_error.json") as f:
    recon = json.load(f)
t_fractions = recon["t_fractions"]

plt.figure(figsize=(8, 5))
for predictor, by_t in recon["mse_by_predictor_and_t"].items():
    xs = [float(t) for t in t_fractions]
    ys = [by_t[str(t)] if str(t) in by_t else by_t[t] for t in t_fractions]
    plt.plot(xs, ys, marker="o", label=predictor)
plt.xlabel("t / T (noise level)")
plt.ylabel(r"MSE($\hat x_0$, $x_0$)")
plt.title(f"Per-timestep reconstruction error @ step={STEP_CAP}")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig2c_reconstruction_mse.png", dpi=150, bbox_inches="tight")
plt.show()

# qualitative complement: what x_hat_0 actually looks like at each noise level, across ALL
# validation images (not just one) -- one real image can be a lucky/unlucky pick, six makes
# the pattern (or lack of one) visible rather than asserted
real_paths = sorted(glob.glob(f"{ANALYSIS_ROOT}/reconstruction/real_x0/*.png"),
                     key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
n_val = len(real_paths)

for val_idx in range(n_val):
    fig, axs = plt.subplots(3, len(t_fractions) + 1, figsize=(3 * (len(t_fractions) + 1), 9))
    for row, predictor in enumerate(["noise", "x0", "mean"]):
        axs[row, 0].imshow(Image.open(real_paths[val_idx]))
        axs[row, 0].axis("off")
        axs[row, 0].set_title("real $x_0$" if row == 0 else "")
        axs[row, 0].text(-0.4, 0.5, predictor, fontsize=13, rotation=90,
                          va="center", ha="center", transform=axs[row, 0].transAxes)
        for col, t in enumerate(t_fractions):
            p = f"{ANALYSIS_ROOT}/reconstruction/{predictor}/t={t}/{val_idx}.png"
            axs[row, col + 1].imshow(Image.open(p))
            axs[row, col + 1].axis("off")
            if row == 0:
                axs[row, col + 1].set_title(f"t/T={t}")

    plt.suptitle(rf"$\hat x_0$ at each noise level, per predictor (validation image {val_idx}/{n_val - 1})", fontsize=15)
    plt.tight_layout()
    plt.savefig(f"fig2d_reconstruction_images_val{val_idx}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Figure + Table — FID vs. training step (all checkpoints, all configs)

The result only possible because every checkpoint was kept, not just the final one: shows *when* each config actually converges. Precomputed by `run_analysis.py`/`analyze_checkpoint.py` at evenly-spaced checkpoints per config (100 samples each — enough to show the trend, not publication-grade) plus `STEP_CAP` (500 samples, the headline number). **`STEP_CAP` uses a different, larger sample count than the rest of the curve** — it's marked with a star and excluded from the "first step below threshold" table below, so it isn't mistaken for just another curve point. Some configs have curve points past `STEP_CAP` (they were trained further) — shown for context, not used in any comparison.

In [ ]:
plt.figure(figsize=(11, 6))
for label, curve in fid_curves.items():
    steps_ = list(curve.keys())
    fids_ = list(curve.values())
    line, = plt.plot(steps_, fids_, marker="o", markersize=5, label=label)
    # STEP_CAP is the headline comparison point (500 samples); other points used 100 -- mark
    # it distinctly so it isn't read as just another point on the same trend. Some configs
    # have points past STEP_CAP (they trained further) -- shown for context, not compared on.
    plt.plot(STEP_CAP, curve[STEP_CAP], marker="*", markersize=16,
              color=line.get_color(), markeredgecolor="black", linestyle="none")
plt.axvline(STEP_CAP, color="gray", linestyle="--", alpha=0.4, linewidth=1)
plt.xlabel("Training step")
plt.ylabel("FID (lower is better)")
plt.yscale("log")
plt.title(f"FID vs. training step, all 5 configs\n(★ = step_cap={STEP_CAP}, 500 samples; ● = curve point, 100 samples)")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.savefig("fig3_fid_vs_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Convergence-speed table: earliest step each config's FID first dropped below a threshold.
# Uses only the 100-sample curve points (excludes the 500-sample STEP_CAP point) so the
# threshold comparison is apples-to-apples across every row.
conv_rows = []
for label, curve in fid_curves.items():
    curve_only = {s: f for s, f in curve.items() if s != STEP_CAP}
    first_below_20 = next((s for s, f in sorted(curve_only.items()) if f < 20), None)
    first_below_15 = next((s for s, f in sorted(curve_only.items()) if f < 15), None)
    best_step, best_fid = min(curve_only.items(), key=lambda sf: sf[1])
    conv_rows.append({
        "label": label,
        "first_step_FID<20": first_below_20,
        "first_step_FID<15": first_below_15,
        "best_curve_step": best_step,
        "best_curve_FID": round(best_fid, 4),
        "step_cap_FID": round(curve[STEP_CAP], 4),
    })
df_convergence = pd.DataFrame(conv_rows)
df_convergence

## 5. Figure — sample quality progression (linear + noise)

The longest, most-checkpointed run — visual complement to the FID curve above, showing the same convergence directly in image space. Precomputed by `run_analysis.py` at ~5 evenly-spaced steps, synced to `gdrive:lab1-analysis/progression/step={step}/`.

In [ ]:
prog_dirs = sorted(glob.glob(f"{ANALYSIS_ROOT}/progression/step=*"),
                    key=lambda d: int(re.search(r"step=(\d+)", d).group(1)))
chosen_steps = [int(re.search(r"step=(\d+)", d).group(1)) for d in prog_dirs]

fig, axs = plt.subplots(len(prog_dirs), 8, figsize=(24, 3 * len(prog_dirs)))
if len(prog_dirs) == 1:
    axs = axs[None, :]
for row, (d, step) in enumerate(zip(prog_dirs, chosen_steps)):
    paths = sorted(glob.glob(f"{d}/*.png"), key=lambda p: int(re.search(r"(\d+)\.png$", p).group(1)))
    load_png_grid(paths, axs[row])
    axs[row, 0].text(-0.3, 0.5, f"step {step}", fontsize=14, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("linear + noise: sample quality over training", fontsize=16)
plt.tight_layout()
plt.savefig("fig4_progression.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Figure — training loss curves overlaid

Pulled directly from each config's `train_state.pt` (`losses` list, saved throughout training), smoothed with a moving average since raw per-step MSE loss is noisy.

In [ ]:
plt.figure(figsize=(11, 6))
WINDOW = 200
for label in CONFIGS:
    d = ckpt_dir(label)
    ts = torch.load(f"{d}/train_state.pt", map_location="cpu", weights_only=False)
    losses = np.array(ts["losses"])
    if len(losses) >= WINDOW:
        smoothed = np.convolve(losses, np.ones(WINDOW) / WINDOW, mode="valid")
    else:
        smoothed = losses
    plt.plot(smoothed, label=label, alpha=0.85)
plt.xlabel("Training step")
plt.ylabel(f"Loss ({WINDOW}-step moving average)")
plt.title("Training loss curves, all 5 configs")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

**On `linear_mean`'s loss curve — correction:** an earlier version of this note claimed only 2 isolated early outliers based on a stale snapshot of `train_state.pt` (fetched before training fully extended to its current final step). The actual rendered curve above shows **several** distinct spikes spread across training, not just one early pair — that earlier claim was wrong and has been removed. What's still true and worth reporting: the spikes look transient (the loss returns to baseline each time, not a sustained drift), and `linear_mean`'s poor FID/collapsed samples are independently well-supported by the reconstruction-error results above (§3c) — its `x_hat_0` error is already large at *low* noise (t/T=0.1), which spikes elsewhere in training can't explain on their own. One concrete, checkable hypothesis for the spikes' cause: `linear_mean` was trained across multiple resumed runs (see the `*_extend.log` files, if you still have pod access) — a resume can briefly perturb the optimizer/LR state. Worth checking whether spike step numbers line up with resume boundaries before asserting a cause in the report; if you can't verify it before the deadline, just report the pattern ("several transient loss spikes observed, cause not fully diagnosed") rather than a specific unverified explanation.

## 7. Figure — worst samples per config

FID is a distributional statistic with no per-image value, so "worst sample" needs a proxy: each config's final-checkpoint generations were ranked by nearest-neighbor distance to the real Inception activation cloud (a standard stand-in for "how unrealistic is this one image"). Precomputed by `run_analysis.py`, synced to `gdrive:lab1-analysis/worst_samples/{label}/` with the underlying scores in `error_analysis_{label}.json`. Shows the 6 worst per config.

In [ ]:
N_WORST_SHOWN = 6
fig, axs = plt.subplots(len(CONFIGS), N_WORST_SHOWN, figsize=(3 * N_WORST_SHOWN, 3 * len(CONFIGS)))

for row, label in enumerate(CONFIGS):
    with open(f"{ANALYSIS_ROOT}/error_analysis_{label}.json") as f:
        err = json.load(f)
    worst = err["worst"][:N_WORST_SHOWN]  # already sorted worst-first by run_analysis.py
    paths = [f"{ANALYSIS_ROOT}/worst_samples/{label}/{w['file']}" for w in worst]
    load_png_grid(paths, axs[row])
    for col, w in enumerate(worst):
        axs[row, col].set_title(f"d={w['nn_distance']:.1f}", fontsize=10)
    axs[row, 0].text(-0.3, 0.5, label, fontsize=13, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)

plt.suptitle("Worst samples per config (by nearest-neighbor distance to real Inception features)", fontsize=16)
plt.tight_layout()
plt.savefig("fig6_worst_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save everything for the report

Copies the final table CSVs and all figure PNGs into Drive so they survive the Colab runtime shutting down.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/lab1-analysis"
os.makedirs(OUT_DIR, exist_ok=True)

df_final.to_csv(f"{OUT_DIR}/table1_final_fid.csv", index=False)
df_convergence.to_csv(f"{OUT_DIR}/table2_convergence.csv", index=False)

import shutil
for f in glob.glob("fig*.png"):
    shutil.copy(f, OUT_DIR)

print("Saved to", OUT_DIR)
print(os.listdir(OUT_DIR))